# 02_transformar_ventas
Limpia `ventas_bronze`, crea `ventas_silver` y `ventas_por_categoria`. Se puede ejecutar varias veces sin duplicar datos (modo overwrite).

In [0]:
from pyspark.sql import functions as F
CAT, ESQ = "workspace", "ingenieria_datos_pablo"

bronze = spark.read.table(f"{CAT}.{ESQ}.ventas_bronze")
antes = bronze.count()

sin_duplicados = bronze.dropDuplicates()                      # registros idénticos
limpio = (sin_duplicados
          .filter(F.col("producto").isNotNull() & (F.trim(F.col("producto")) != ""))
          .filter((F.col("cantidad") > 0) & (F.col("precio_unitario") > 0)))
silver = limpio.withColumn("total_venta", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.{ESQ}.ventas_silver")

resumen = (spark.read.table(f"{CAT}.{ESQ}.ventas_silver")
           .groupBy("categoria")
           .agg(F.round(F.sum("total_venta"), 2).alias("valor_total"), F.count("*").alias("num_ventas"))
           .orderBy("categoria"))
resumen.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CAT}.{ESQ}.ventas_por_categoria")

despues = spark.table(f"{CAT}.{ESQ}.ventas_silver").count()
print(f"Registros ANTES de la limpieza (bronze): {antes}")
print(f"Después de quitar duplicados:            {sin_duplicados.count()}")
print(f"Registros DESPUÉS de la limpieza (silver): {despues}")

Registros ANTES de la limpieza (bronze): 16
Después de quitar duplicados:            15
Registros DESPUÉS de la limpieza (silver): 13


In [0]:
display(spark.table(f"{CAT}.{ESQ}.ventas_por_categoria"))

categoria,valor_total,num_ventas
Electrónica,2495.0,6
Muebles,730.0,4
Papelería,61.0,3


In [0]:
# Control de repetición: cada ejecución deja una fila; los conteos deben ser iguales en todas
(spark.createDataFrame([(antes, despues, spark.table(f"{CAT}.{ESQ}.ventas_por_categoria").count())],
                       "registros_bronze INT, registros_silver INT, filas_resumen INT")
 .withColumn("ejecutado_en", F.current_timestamp())
 .write.mode("append").saveAsTable(f"{CAT}.{ESQ}.control_ejecuciones_02"))
display(spark.table(f"{CAT}.{ESQ}.control_ejecuciones_02").orderBy("ejecutado_en"))

registros_bronze,registros_silver,filas_resumen,ejecutado_en
16,13,3,2026-11-14T00:51:27.635Z
16,13,3,2026-11-14T00:51:46.366Z
